# Data Preparation & Validation Strategy

Input: `df = load_clean()` từ `src/data/loader.py` (đã sạch, `customerID` đã bỏ khỏi quá trình xử lý, `SeniorCitizen` đã chuẩn hóa).

Mục tiêu của notebook: tách một holdout test set và ghi ra đĩa để giữ bất biến, chọn chiến lược cross-validation, dựng preprocessing pipeline tái sử dụng được, và tính class-imbalance ratio để dùng khi train.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.churn_classification.data_split import RANDOM_STATE, TEST_SIZE, get_split
from src.churn_classification.preprocessing import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    build_preprocessor,
    compute_scale_pos_weight,
    split_X_y,
)


## 1. Tách train / holdout — tại sao tách riêng khỏi CV

Nếu dùng chung CV score vừa để chọn model/tune hyperparameter vừa để báo cáo performance cuối cùng, con số cuối sẽ lạc quan giả tạo, vì toàn bộ dữ liệu đã được "nhìn" qua nhiều vòng trước khi chốt số. Cách sạch hơn là cắt riêng một holdout test set và không đụng tới nó cho tới lúc đánh giá cuối; mọi việc chọn model/tune chỉ thấy phần train pool còn lại.

Split được ghi ra đĩa (`data/processed/churn_classification/{train,test}.csv`) thay vì gọi lại `train_test_split` mỗi lần chạy. Nếu không, một lần re-run với version sklearn/pandas khác hay thứ tự dòng đổi có thể vô tình đổi holdout, làm khách hàng lẽ ra bị giữ lại lọt vào tập train. `get_split()` tạo lần đầu, các lần sau chỉ load lại y nguyên.

In [2]:
train_df, test_df = get_split()
print(f"train_df: {train_df.shape}  ({TEST_SIZE:.0%} held out -> test_df: {test_df.shape})")
print(f"random_state={RANDOM_STATE}")

print()
print("Ty le Churn - train:")
print(train_df['Churn'].value_counts(normalize=True).round(4))
print("Ty le Churn - test:")
print(test_df['Churn'].value_counts(normalize=True).round(4))


Loading cached split from /home/tthhieu/survival-analysis/data/processed/churn_classification (created earlier). Delete this folder and re-run to regenerate with current TEST_SIZE/RANDOM_STATE.
train_df: (5977, 21)  (15% held out -> test_df: (1055, 21))
random_state=42

Ty le Churn - train:
Churn
No     0.7341
Yes    0.2659
Name: proportion, dtype: float64
Ty le Churn - test:
Churn
No     0.7346
Yes    0.2654
Name: proportion, dtype: float64


In [3]:
# Leak-check dung customerID (dinh danh that), KHONG dung so sanh full-row: bo du lieu
# nhieu cot categorical + it gia tri numeric ung khien nhieu khach hang khac nhau co the
# trung y het profile mot cach ngau nhien -- full-row match se bao dong gia (da gap khi
# thu ban dau). customerID la thu duy nhat dam bao dung "cung 1 khach hang".
customer_overlap = set(train_df["customerID"]) & set(test_df["customerID"])
assert len(customer_overlap) == 0, "Phat hien customerID trung giua train va test!"
print(f"Xac nhan: 0 customerID trung giua train ({len(train_df)}) va test ({len(test_df)}).")

# Thong tin them (khong phai loi): so dong trung profile hoan toan (bo qua customerID)
# giua train/test -- ky vong > 0 vi ly do neu tren, khong anh huong toi validation strategy.
dup_profile = pd.merge(train_df.drop(columns=["customerID"]), test_df.drop(columns=["customerID"]), how="inner")
print(f"(Thong tin) {len(dup_profile)} dong co feature profile trung nhau giua train/test do trung hop ngau nhien, khac customerID -- khong phai leakage.")


Xac nhan: 0 customerID trung giua train (5977) va test (1055).
(Thong tin) 7 dong co feature profile trung nhau giua train/test do trung hop ngau nhien, khac customerID -- khong phai leakage.


## 2. Chiến lược cross-validation

Mỗi khách hàng chỉ xuất hiện một dòng, một thời điểm, không có chiều thời gian và không lặp lại (`customerID` unique 100%), nên không cần time-series split hay group-aware split. `StratifiedKFold` là lựa chọn hợp lý: giữ nguyên tỷ lệ 73/27 ở mỗi fold. Với mức imbalance này, một fold ngẫu nhiên không stratify có thể lấy quá ít mẫu Churn=Yes, làm PR-AUC của fold đó nhiễu mạnh chỉ vì cỡ mẫu dương nhỏ.

In [4]:
X_train, y_train = split_X_y(train_df)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Kiem tra ty le lop duong (Churn=Yes) o tung fold (phai xap xi 0.265 o moi fold):")
for i, (_, val_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"  fold {i}: n={len(val_idx):4d}  pos_rate={y_train.iloc[val_idx].mean():.4f}")


Kiem tra ty le lop duong (Churn=Yes) o tung fold (phai xap xi 0.265 o moi fold):
  fold 0: n=1196  pos_rate=0.2659
  fold 1: n=1196  pos_rate=0.2659
  fold 2: n=1195  pos_rate=0.2653
  fold 3: n=1195  pos_rate=0.2661
  fold 4: n=1195  pos_rate=0.2661


## 3. Khai báo kiểu feature & một cấu trúc dữ liệu đáng chú ý

Danh sách numeric/categorical được khai báo tường minh trong `src/churn_classification/preprocessing.py` thay vì dùng `df.select_dtypes` tự động, để pipeline không âm thầm đổi hành vi nếu một cột đổi dtype ở lần load sau.

In [5]:
# customerID van con trong train_df/test_df (giu de truy vet), nhung khong phai
# feature -> loai tuong minh khoi phep so sanh nay, khong dua vao split_X_y() de "tu dong" loai.
assert set(NUMERIC_FEATURES + CATEGORICAL_FEATURES) == set(train_df.columns) - {"Churn", "customerID"}
print(f"{len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical -> khop du 20 feature + Churn + customerID")

for c in CATEGORICAL_FEATURES:
    print(f"{c:20s} n_unique={train_df[c].nunique()}  {sorted(train_df[c].unique().tolist())}")


3 numeric, 16 categorical -> khop du 20 feature + Churn + customerID
gender               n_unique=2  ['Female', 'Male']
SeniorCitizen        n_unique=2  ['No', 'Yes']
Partner              n_unique=2  ['No', 'Yes']
Dependents           n_unique=2  ['No', 'Yes']
PhoneService         n_unique=2  ['No', 'Yes']
MultipleLines        n_unique=3  ['No', 'No phone service', 'Yes']
InternetService      n_unique=3  ['DSL', 'Fiber optic', 'No']
OnlineSecurity       n_unique=3  ['No', 'No internet service', 'Yes']
OnlineBackup         n_unique=3  ['No', 'No internet service', 'Yes']
DeviceProtection     n_unique=3  ['No', 'No internet service', 'Yes']
TechSupport          n_unique=3  ['No', 'No internet service', 'Yes']
StreamingTV          n_unique=3  ['No', 'No internet service', 'Yes']
StreamingMovies      n_unique=3  ['No', 'No internet service', 'Yes']
Contract             n_unique=3  ['Month-to-month', 'One year', 'Two year']
PaperlessBilling     n_unique=2  ['No', 'Yes']
PaymentMethod      

Một điểm đáng chú ý: 6 cột (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) dùng giá trị `"No internet service"` thay cho `"No"` bất cứ khi nào `InternetService == "No"`. Nghĩa là tại đúng những dòng đó, 6 cột này collinear hoàn hảo với nhau và với `InternetService`; sau one-hot, 6 dummy `"...No internet service"` sẽ giống hệt nhau.

Không cần xử lý đặc biệt ở đây. `LogisticRegression` mặc định của sklearn dùng L2 regularization nên xử lý collinearity ổn định về số học (chia đều trọng số giữa các cột giống nhau), còn tree-based model thì không bị ảnh hưởng bởi collinearity. Cái giá là 6 hệ số hồi quy đó khó diễn giải riêng lẻ — ghi lại để lúc đọc coefficient/feature importance không hiểu sai.

## 4. Preprocessing pipeline (ColumnTransformer)

Chi tiết lựa chọn (median imputer, StandardScaler, OneHotEncoder với `drop='if_binary'`) đã ghi trong docstring của `build_preprocessor()`. Ở đây chỉ fit thử trên `X_train` để xác nhận pipeline chạy được và xem shape đầu ra. Không fit trên toàn bộ `df` để tránh leakage.

In [6]:
preprocessor = build_preprocessor()
X_train_transformed = preprocessor.fit_transform(X_train)

print(f"X_train: {X_train.shape} -> sau preprocessing: {X_train_transformed.shape}")
print(f"So chieu tang do OneHotEncoder: {X_train_transformed.shape[1] - X_train.shape[1]} cot moi")
assert not np.isnan(X_train_transformed).any(), "Con NaN sau preprocessing!"
print("Xac nhan: khong con NaN sau preprocessing.")


X_train: (5977, 19) -> sau preprocessing: (5977, 40)
So chieu tang do OneHotEncoder: 21 cot moi
Xac nhan: khong con NaN sau preprocessing.


## 5. Xử lý class imbalance

Ratio neg/pos tính chỉ trên `y_train`, không phải toàn bộ `df` hay test set. Dù chỉ là một con số tỷ lệ, tính trên test set vẫn là một dạng leakage (thông tin phân phối của test set rò rỉ vào cấu hình training).

In [7]:
scale_pos_weight = compute_scale_pos_weight(y_train)
print(f"scale_pos_weight (neg/pos) tren train pool: {scale_pos_weight:.4f}")


scale_pos_weight (neg/pos) tren train pool: 2.7615


Chọn `class_weight='balanced'` (LogisticRegression/RandomForest — sklearn tự tính) hoặc `scale_pos_weight` (XGBoost/LightGBM — truyền thủ công con số vừa tính), không dùng SMOTE. Lý do:

1. SMOTE nội suy giữa các điểm lân cận trong không gian feature đã one-hot. Với dữ liệu phần lớn categorical, nội suy tuyến tính giữa các one-hot vector tạo ra điểm "không tồn tại trong thực tế", dễ overfit vào artefact của thuật toán resampling hơn là học pattern churn thật.
2. `class_weight`/`scale_pos_weight` chỉ đổi trọng số trong loss, không tạo/xóa dữ liệu — đơn giản hơn, không cần kéo `imblearn.pipeline.Pipeline` vào để tránh resampling leak qua fold.
3. Vốn đã quyết định xử lý cost-sensitivity ở tầng metric/threshold (F2), nên chỉnh trọng số ở tầng loss là cách nhất quán để model "biết" trước rằng bỏ sót lớp Yes tốn kém hơn.

Nếu ở bước modeling mà `class_weight` không đủ cải thiện recall lớp Yes thì mới quay lại cân nhắc SMOTE, không dùng cả hai cùng lúc để còn quy được kết quả về đâu.